In [15]:
import math
from itertools import product

In [16]:
# -------------------------------------------------------------------------------------------
# Input data
# -------------------------------------------------------------------------------------------
# Annual demand for each product
D = [1000, 300, 100, 50]

# Product-specific ordering costs
k_spec = [20, 25, 30, 50]

# Common (joint) ordering cost
K_common = 150

# Unit costs
c = [50, 60, 30, 30]

# Holding cost rate
h_rate = 0.15

# Per-unit annual holding costs H_i = h * c_i
H = [h_rate * ci for ci in c]

n_products = len(D)

In [17]:
# -------------------------------------------------------------------------------------------
# Strategy 1: Products are sourced independently
# -------------------------------------------------------------------------------------------
def cost_independent():
    """
    Each product is ordered independently.
    Each order of product i pays: K_common + k_spec[i]
    """
    Q_opt = []
    total_cost = 0.0

    for Di, ki, Hi in zip(D, k_spec, H):
        S_i = K_common + ki  # total fixed cost per order for product i
        Qi = math.sqrt(2 * Di * S_i / Hi)  # EOQ
        Q_opt.append(Qi)

        ordering_cost = Di / Qi * S_i
        holding_cost = Qi / 2 * Hi
        total_cost += ordering_cost + holding_cost

    return Q_opt, total_cost

In [18]:
# -------------------------------------------------------------------------------------------
# Strategy 2: All four products are sourced with the same frequency
# -------------------------------------------------------------------------------------------
def cost_same_frequency():
    """
    All products share the same reorder interval T.
    Each joint order pays: K_common + sum(k_spec).
    Q_i = D_i * T for each product.
    """
    A = sum(Hi * Di for Hi, Di in zip(H, D))                  # coefficient on T
    K_total = K_common + sum(k_spec)                          # joint fixed cost
    T = math.sqrt(2 * K_total / A)                            # optimal T

    Q = [Di * T for Di in D]                                  # lot sizes
    ordering_cost = K_total / T
    holding_cost = sum(Qi * Hi / 2 for Qi, Hi in zip(Q, H))
    total_cost = ordering_cost + holding_cost

    return T, Q, total_cost


In [19]:
# -------------------------------------------------------------------------------------------
# Strategy 3: Order frequencies are determined according to the tailored aggregation strategy
# -------------------------------------------------------------------------------------------
def cost_tailored(m):
    """
    Tailored aggregation.
    m[i] = integer multiple for product i, tau_i = m_i * t is its reorder interval.
    Assumption: at least one product has m_i = 1, so there is one base order every t.

    Annual cost:
      holding  = sum( H_i * D_i * tau_i / 2 )
      spec_ord = sum( k_i / tau_i )
      common   = K_common / t
    """
    m = list(m)
    if 1 not in m:
        raise ValueError("At least one product must have m_i = 1 so that there's a base cycle.")

    # Coefficients a * t + b / t
    A = 0.5 * sum(Hi * Di * mi for Hi, Di, mi in zip(H, D, m))
    b = K_common + sum(ki / mi for ki, mi in zip(k_spec, m))

    t = math.sqrt(b / A)                     # optimal base cycle length
    tau = [mi * t for mi in m]               # individual reorder intervals
    Q = [Di * ti for Di, ti in zip(D, tau)]  # lot sizes

    holding_cost = sum(Qi * Hi / 2 for Qi, Hi in zip(Q, H))
    spec_cost = sum(ki / (mi * t) for ki, mi in zip(k_spec, m))
    common_cost = K_common / t
    total_cost = holding_cost + spec_cost + common_cost

    return t, tau, Q, total_cost

In [20]:
# -------------------------------------------------------------------------------------------
# Strategy 3 expanded: search over a grid of m_i to find best TAS pattern
# -------------------------------------------------------------------------------------------
def search_best_tailored(max_multiple=4):
    """
    Try all m_i in {1, ..., max_multiple}, require at least one m_i = 1.
    Return the pattern with minimum cost.
    """
    best = None
    best_pattern = None
    best_details = None

    for m in product(range(1, max_multiple + 1), repeat=n_products):
        if 1 not in m:
            continue  # must have at least one base product

        t, tau, Q, total_cost = cost_tailored(m)

        if best is None or total_cost < best:
            best = total_cost
            best_pattern = m
            best_details = (t, tau, Q)

    return best_pattern, best, best_details

In [21]:
# -------------------------------------------------------------------------------------------
# Final Run and Report
# -------------------------------------------------------------------------------------------
if __name__ == "__main__":
    # Strategy 1: Independent
    Q_ind, cost_ind = cost_independent()
    print("1) Independent sourcing")
    print("   EOQs per product:", [round(q, 2) for q in Q_ind])
    print("   Annual cost     :", round(cost_ind, 2))
    print()

    # Strategy 2: Same frequency
    T_same, Q_same, cost_same = cost_same_frequency()
    print("2) Same frequency for all products")
    print("   Common T        :", round(T_same, 4), "years")
    print("   Lot sizes       :", [round(q, 2) for q in Q_same])
    print("   Annual cost     :", round(cost_same, 2))
    print()

    # Strategy 3: Tailored aggregation, given pattern (1,1,2,3)
    m_given = (1, 1, 2, 3)
    t_given, tau_given, Q_given, cost_given = cost_tailored(m_given)
    print("3a) Tailored aggregation (pattern m = {})".format(m_given))
    print("    Base cycle t   :", round(t_given, 4), "years")
    print("    Intervals tau  :", [round(x, 4) for x in tau_given])
    print("    Lot sizes      :", [round(q, 2) for q in Q_given])
    print("    Annual cost    :", round(cost_given, 2))
    print()

    # Strategy 3 expanded: search over a grid of m_i to find best TAS pattern
    best_m, best_cost, (t_best, tau_best, Q_best) = search_best_tailored(max_multiple=4)
    print("3b) Best TAS pattern found in search (m_i in {1,...,4})")
    print("    Best pattern m :", best_m)
    print("    Base cycle t   :", round(t_best, 4), "years")
    print("    Intervals tau  :", [round(x, 4) for x in tau_best])
    print("    Lot sizes      :", [round(q, 2) for q in Q_best])
    print("    Annual cost    :", round(best_cost, 2))

1) Independent sourcing
   EOQs per product: [212.92, 108.01, 89.44, 66.67]
   Annual cost     : 3271.48

2) Same frequency for all products
   Common T        : 0.2249 years
   Lot sizes       : [224.89, 67.47, 22.49, 11.24]
   Annual cost     : 2445.66

3a) Tailored aggregation (pattern m = (1, 1, 2, 3))
    Base cycle t   : 0.1962 years
    Intervals tau  : [0.1962, 0.1962, 0.3924, 0.5886]
    Lot sizes      : [196.21, 58.86, 39.24, 29.43]
    Annual cost    : 2310.41

3b) Best TAS pattern found in search (m_i in {1,...,4})
    Best pattern m : (1, 1, 2, 3)
    Base cycle t   : 0.1962 years
    Intervals tau  : [0.1962, 0.1962, 0.3924, 0.5886]
    Lot sizes      : [196.21, 58.86, 39.24, 29.43]
    Annual cost    : 2310.41


In [22]:
# -------------------------------------------------------------------------------------------
# Report the cost of the tailored aggregation strategy in the text field.
# -------------------------------------------------------------------------------------------
print("\nOptimal Tailored Aggregation Strategy Cost:", round(best_cost, 2))


Optimal Tailored Aggregation Strategy Cost: 2310.41
